# Visual-cell pipeline - single-session walkthrough

**Session: LDR004 `ltdk_10`** (left hemisphere -> right visual field).

This notebook visualizes, one step at a time, how a cell is decided to be **visual** and how its
**egocentric receptive field (RF)** - preferred wall *direction* and *distance* - is located. It reuses
the existing modules (`visual_cells`, `receptive_field`, `cell_explorer`) and the already-computed
`summary.pkl` for this session.

**Pipeline**
1. **Visual classification** - reliable egocentric map in light, *not* in dark.
2. **Egocentric rate-map** - occupancy-normalized, 50 cm cap, occupancy floor, smoothing.
3. **RF localization** - two-peak front/back confound, keep the visible member.
4. **Hemisphere / FOV selection** - left hemisphere => right field; reject rear ghosts.
5. **Split-half stability** - the RF must not move between session halves.
6. **Keep set & population** - `keep = is_visual & rf_frontal & rf_stability < 30`.

## Setup

Load the session pickle and point at its results folder.

In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, r"c:\Users\acostamendoza\projects\distance-project\mini2p-analysis")
from cell_explorer import CellExplorer
from visual_cells import VisualClassifier
from receptive_field import RFAnalyzer
from ego_reformat import compute_ego_maps

EBR = r"D:\data-distance-coding-project\mini2p-data\LDR004\ebr_pipeline_251204_LDR_LDR004_ltdk_10.pkl"
RESULTS = r"D:\data-distance-coding-project\mini2p-data\LDR004\results\LDR004_ltdk_10"
SUMMARY_PKL = os.path.join(RESULTS, "summary.pkl")   # per-cell results already computed for this session

ce = CellExplorer(EBR)
n_rays = 360 // ce.settings["ego_rays_deg"]
n_dist = round(ce.settings["ego_radius"] / ce.settings["ego_bin_size"])
print(ce.session_id, "|", ce.n_cells, "cells")
print(f"ego map grid: {n_rays} rays x {n_dist} dist bins "
      f"(bin {ce.settings['ego_bin_size']*100:.1f} cm, session radius {ce.settings['ego_radius']*100:.0f} cm)")

c:\Users\acostamendoza\projects\distance-project\mini2p-analysis\cell_explorer.py:28: UserWarning: This figure was saved with matplotlib version 3.9.2 and loaded with 3.9.4 so may not function correctly.
  ebr = pickle.load(f)


251204_LDR_LDR004_ltdk_10 | 418 cells
ego map grid: 120 rays x 14 dist bins (bin 2.5 cm, session radius 35 cm)


In [ ]:
# Example cells for the walkthrough (chosen earlier on this session).
EX_GOOD     = 38     # clean visual cell: sharp frontal RF, stable across halves
EX_UNSTABLE = 208    # visual but unstable: RF location jumps between halves -> dropped

# Pull a ghost-pair and an ambiguous example from the already-computed summary (fallbacks if absent).
EX_GHOST, EX_AMBIG = EX_GOOD, None
if os.path.exists(SUMMARY_PKL):
    summ = pd.read_pickle(SUMMARY_PKL)
    g = summ[summ["rf_frontal"] & summ["rf_ghost_angle_deg"].notna()]
    if len(g):
        EX_GHOST = int(g.loc[g["rf_ghost_angle_deg"].abs().idxmax(), "cell"])   # clearest rear ghost
    a = summ[summ["rf_ambiguous"].fillna(False)]
    if len(a):
        EX_AMBIG = int(a.iloc[0]["cell"])
else:
    summ = None
print("examples:", dict(good=EX_GOOD, unstable=EX_UNSTABLE, ghost=EX_GHOST, ambiguous=EX_AMBIG))

## The data: egocentric boundary maps

For every frame we cast 120 rays from the animal and record the distance to the nearest wall along
each. Binning spikes by (ray angle, wall distance) and normalizing by occupancy gives each cell an
**egocentric map**: how it fires as a function of where a wall sits relative to the animal's heading
(front = up). A visual boundary cell shows a bright spot at its preferred (direction, distance).

In [ ]:
# Raw per-cell view (argmax preferred): trajectory+spikes | egocentric map | angular | distance,
# for light (top row) and dark (bottom row).
ce.plot_cell(EX_GOOD)
plt.show()

## Step 1 - Visual classification

A cell is **visual** if its egocentric map is *reliable* (reproducible across time) in the **light**
but **not** in the **dark**:

`is_visual = light_reliable AND (light_reliability >= 0.5) AND NOT dark_reliable`

Reliability = mean split-half correlation of the map across balanced partitions of the session.
"Reliable" = the real value beats the 99th percentile of a **circular-shift null** (the cell's activity
rolled out of register with the wall geometry). Requiring "not in the dark" removes cells that code
boundaries without vision, and the light>dark firing-rate confound.

In [ ]:
clf = VisualClassifier(ce)
vis_df = clf.classify(verbose=True)    # split-half reliability vs circular-shift null (~1-2 min)
vis_df.head()

In [ ]:
# Per-cell evidence: light map | dark map | reliability vs shuffle-null histograms (real lines + p99).
# A non-visual contrast: reliable in BOTH light and dark (codes boundaries with lights off too) -> rejected.
both = vis_df[vis_df["light_sig"] & vis_df["dark_sig"]]
EX_NONVIS = int(both.iloc[0]["cell"]) if len(both) else int(vis_df[~vis_df["is_visual"]].iloc[0]["cell"])

clf.plot_cell(EX_GOOD)       # VISUAL: reliable in light, null in dark
plt.show()
clf.plot_cell(EX_NONVIS)     # not visual (reliable in dark too, or below the 0.5 floor)
plt.show()
print("non-visual example:", EX_NONVIS)

In [ ]:
# Whole-population view of the decision.
fig, ax = plt.subplots(figsize=(5.6, 5.2))
col = np.where(vis_df["is_visual"], "tab:red", "lightgray")
ax.scatter(vis_df["light_reliability"], vis_df["dark_reliability"], c=col, s=14, edgecolors="none")
ax.axvline(0.5, ls="--", color="k", lw=1)
ax.text(0.51, ax.get_ylim()[1], " light floor 0.5", color="k", fontsize=8, va="top")
ax.set_xlabel("light split-half reliability"); ax.set_ylabel("dark split-half reliability")
ax.set_title(f"visual = light-reliable & dark-unreliable  "
             f"(n={int(vis_df['is_visual'].sum())} of {len(vis_df)})")
plt.show()

## Step 2 - Building the egocentric rate-map

The map fed to the RF stage differs from the raw pipeline map in three ways:
- **50 cm cap** (not 35 cm, not the full ~73.5 cm box) - recovers the field without the far,
  barely-visited bins where `spk/occ` explodes.
- **occupancy floor** (`min_occ`) - blanks bins visited by too few frames.
- **NaN-aware Gaussian smoothing** (`sigma = 0.5` bins) - the angular axis wraps at front; holes stay holes.

In [ ]:
rf = RFAnalyzer(ce, clf)     # defaults: ego_radius 0.5 (50 cm), min_occ 0, smooth 0.5, fov (-120, 120)

# occupancy | raw 50-cm map | smoothed map (what the RF is computed from), for the example cell.
s50 = dict(ce.settings); s50["ego_radius"] = 0.50
ego_raw, occ = compute_ego_maps(ce.df, s50, 0, return_occ=True)   # light = 0
ego_sm = rf._ego("light")                                         # 50 cm + Gaussian smooth (sigma 0.5 bins)

fig = plt.figure(figsize=(15, 4.6))
panels = [("occupancy (frames / bin)", occ, "viridis"),
          (f"raw map, 50 cm - cell {EX_GOOD}", ego_raw[EX_GOOD], "Blues"),
          (f"smoothed map (sigma 0.5) - cell {EX_GOOD}", ego_sm[EX_GOOD], "Blues")]
for i, (title, M, cmap) in enumerate(panels):
    ax = fig.add_subplot(1, 3, i + 1, projection="polar")
    CellExplorer._draw_ego(ax, M, cmap, title)
plt.show()

In [ ]:
# Why 50 cm: 35 cm clips the field; the far box edge is barely visited (occupancy ~1 -> rate blows up).
# Same cell, RF recomputed at a 35 cm vs 50 cm cap (full circle so a rear peak would show).
ce.compare_rf(EX_GOOD, variants=[{"ego_radius": 0.35}, {"ego_radius": 0.50}],
              labels=["35 cm (old cap)", "50 cm (used)"], angle_clip=None)
plt.show()

## Step 3 - Locating the RF (the front/back confound)

In a closed symmetric box a near-wall cell fires for the wall **in front** *and* for its **far-wall
ghost ~180 deg behind** at the complementary distance (the two distances sum to the box size). So we:
1. detect the **two strongest, opposite 2D peaks** of the smoothed map,
2. characterize each with a half-max-weighted **centroid** (angle + vector strength `R`, distance + width),
3. confirm they form a **confound pair** (~180 deg apart) and report the **visible** member as the RF,
   the other as the **ghost** (gray x).

The panels below: trajectory | ego map with RF center (red o) + half-max field + FOV lines + ghost |
angular profile (red = RF center +/- SD, gray dashed = ghost) | distance profile over the RF's run.

In [ ]:
rf.plot_cell(EX_GOOD)
plt.show()

## Step 4 - Hemisphere / FOV selection

Recording is from the **left hemisphere -> right visual field**; signed angle is front = 0, right +,
left -. The field of view is **+/-120 deg**. From a confound pair we keep the member **inside** the FOV
(a rear ghost with |angle| > 120 is dropped). A lateral **+/-90 deg mirror pair** the symmetric box
cannot physically resolve is kept on the **right** (the hemisphere's priority side) and flagged
`rf_ambiguous`.

In [ ]:
# A front/back confound pair: the rear ghost (gray x, |angle| > 120) is rejected, the frontal member kept.
rf.plot_cell(EX_GHOST)
plt.show()
if EX_AMBIG is not None:
    rf.plot_cell(EX_AMBIG)     # lateral mirror pair: kept on the right, flagged AMBIGUOUS
    plt.show()
else:
    print("no rf_ambiguous cell in this session")

## Step 5 - Split-half stability

A real visual RF should sit in the same place all session. We recompute the RF on the **1st** and
**2nd** chronological halves and measure how far the angle moves (`rf_stability`, deg). Well-localized
cells barely move (a few deg); broad / confound-flipping cells swing wildly. **Cut-off 30 deg** - if the
RF is not stable, the cell is not counted as visual.

In [ ]:
ce.compare_rf(EX_GOOD,     split_half=True, angle_clip=None)   # stable: small delta-angle
plt.show()
ce.compare_rf(EX_UNSTABLE, split_half=True, angle_clip=None)   # unstable: large delta-angle -> dropped
plt.show()

## Step 6 - The keep set and the population

The analysis-final visual set is

`keep = is_visual & rf_frontal & (rf_stability < 30)`

read straight from the session's `summary.pkl` (one row per cell, all metrics). The population plot
shows the RF centers of the keep cells: polar scatter (color = `R`, size = reliability),
preferred-angle rose, preferred-distance histogram.

In [ ]:
summ = pd.read_pickle(SUMMARY_PKL)     # already-computed per-cell table for this session
print(f"cells {len(summ)} | visual {int(summ['is_visual'].sum())} | "
      f"visual & frontal {int((summ['is_visual'] & summ['rf_frontal']).sum())} | "
      f"keep (visual & frontal & stability<30) {int(summ['keep'].sum())}")
rf.plot_population(summ[summ["keep"]])
plt.show()

In [ ]:
# The metrics behind the verdict, for the example cells.
cols = ["cell", "is_visual", "light_reliability", "rf_angle_deg", "rf_dist_cm", "rf_R",
        "rf_frontal", "rf_stability", "rf_ghost_angle_deg", "rf_ambiguous", "keep"]
summ.loc[summ["cell"].isin([EX_GOOD, EX_GHOST, EX_UNSTABLE]), cols].round(2)

## Step 7 - Saved outputs & reproducing every session

`session_summary.py` writes, per session, `summary.pkl` + `plot_population.png` +
`visual_cells_rf.pdf`, and a combined `LDR004_all_sessions_summary.pkl` across all 24 sessions.

In [ ]:
# Reproduce the saved outputs (uncomment to run; summarize_all over all sessions takes ~1 hour):
# from session_summary import process_session, summarize_all
# process_session(EBR, RESULTS)                                   # just this session
# summarize_all(os.path.dirname(EBR), os.path.dirname(RESULTS))   # all 24 sessions + combined master

files = sorted(os.listdir(RESULTS)) if os.path.isdir(RESULTS) else ["(folder not found)"]
print("saved for this session ->", RESULTS)
for f in files:
    print("   ", f)